# Déploiement sur Kubernetes

L'API de production tourne sur un cluster GKE, prévu pour monter en charge.
Ce notebook prépare le cluster `purchase-predict-api`, l'espace de noms `api-purchase` et les manifestes.
Le tag d'image est la macro `DOCKER_TAG`, remplacée au build par le `SHORT_SHA`.
Cloud Build construit l'image, rend les YAML et les applique sur le cluster.
L'IP d'exemple de l'Ingress est à remplacer. La requête HTTP suppose un cluster en service.


## Cluster et espace de noms

Créer le cluster `purchase-predict-api` avec le pool par défaut : autoscaling de 1 à 5 nœuds, machines `e2-medium`, disque de 10 Go par nœud. Le plafond de 5 nœuds borne le cluster. Se connecter ensuite avec la commande `gcloud` indiquée dans la console GKE.

Les espaces de noms isolent les ressources. Un cluster neuf contient `default`, `kube-node-lease`, `kube-public` et `kube-system`. Le préfixe `kube` est réservé au cluster. L'API utilise `api-purchase`. Les quotas GKE par défaut suffisent. `ns.yaml` reste à la racine du dépôt : le dossier `k8s/` ne contient que Deployment, Service et Ingress.


## Manifestes

Se placer sur `master` et fusionner `staging` avant d'ajouter les manifestes de production. Le Deployment demande 2 réplicas, 0,5 CPU et 0,5 Go, avec une limite à 1 CPU et 1 Go.

L'image versionnée est `gcr.io/<PROJET_GCP>/purchase-predict-api:DOCKER_TAG`. Le dépôt `<PROJET_GCP>`, `ENV=staging` et `MLFLOW_SERVER=http://<IP_MLFLOW>/` sont à aligner sur le registre du projet, l'environnement et l'adresse MLflow. `DOCKER_TAG` est remplacé au build. Un `kubectl apply` direct sur ce fichier exige un tag réel à la place de la macro.

Sans `--namespace`, kubectl lit l'espace `default`. Le second pod peut démarrer plus tard si l'autoscaling ajoute un nœud. L'Ingress reçoit son adresse après quelques minutes.


## Requête de contrôle

La commande `kubectl get ingress --namespace api-purchase` affiche l'adresse à mettre à la place de `<IP_INGRESS>`. Ne pas relancer sans cluster.


In [ ]:
import os
import requests
import pandas as pd

dataset = pd.read_csv(os.path.expanduser("data/primary.csv"))
dataset = dataset.drop(["user_session", "user_id", "purchased"], axis=1)

requests.post(
    "http://<IP_INGRESS>/predict",  # Remplacer par l'adresse de l'Ingress. Ne pas relancer sans cluster.
    json=dataset.sample(n=10).to_json()
).json()


## Build de production de l'API

Sur `master`, le build de l'API s'arrête au registre, puis déploie sur GKE. `sed` concatène `k8s/*.yaml` en les séparant par `---`, puis remplace `DOCKER_TAG` par `$SHORT_SHA`. Un seul `config.yaml` porte le tag construit et sert au `kubectl apply`.

`CLOUDSDK_COMPUTE_ZONE` et `CLOUDSDK_CONTAINER_CLUSTER` indiquent à l'image `kubectl` la zone `europe-north1-b` et le cluster `purchase-predict-api`. Autoriser Cloud Build sur GKE avant le premier build. Le déclencheur suit la branche `master` du dépôt de l'API.
